# Домашнє завдання: Регуляризація та аналіз результатів моделі

## Бізнес-контекст
Компанія HealthRisk Analytics завершує пілотний етап розробки моделі, яка прогнозує ймовірність хронічних захворювань на підставі медичних даних.
Тепер необхідно перевірити стійкість моделі на реальному наборі даних і оцінити вплив регуляризації на точність і стабільність прогнозів.

## Завдання 1
Підготуйте дані для аналізу:
- Використовуйте датасет про діабет із бібліотеки `sklearn.datasets.load_diabetes`.
- Розділіть дані на навчальну і тестову вибірки в пропорції 80/20 (параметр `random_state=42`).
- Масштабуйте ознаки за допомогою `StandardScaler`.
- Перевірте форму матриць навчальної та тестової вибірок.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch

# Завантаження даних
diabetes = load_diabetes()
X = diabetes.data
y = diabetes.target

# Розділення на навчальну і тестову вибірки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Масштабування ознак
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Форма матриці X_train: {X_train_scaled.shape}")
print(f"Форма матриці X_test: {X_test_scaled.shape}")

# Перетворення у тензори PyTorch
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

## Завдання 2 та 3
**Побудуйте модель MLP для регресії:**
- Вхідний шар: число нейронів = кількості ознак.
- Приховані шари: 64 (ReLU) -> 32 (ReLU) -> 16 (ReLU).
- Вихідний шар: 1 нейрон (без активації).
- Функція втрат: MSELoss.
- Оптимізатор: Adam, lr=0.001.
- Епохи: 50.

**Додайте регуляризацію і порівняйте результати:**
- Проведіть 2 експерименти: без регуляризації та з регуляризацією (L2, `weight_decay=1e-4` в Adam).
- Для кожної обчисліть `MAE` та `R²` на тестовій вибірці.
- Оформіть результати в таблиці.

In [ ]:
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_absolute_error, r2_score
from IPython.display import display

input_dim = X_train_t.shape[1]

# Функція для створення архітектури моделі
def create_model():
    return nn.Sequential(
        nn.Linear(input_dim, 64),
        nn.ReLU(),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, 16),
        nn.ReLU(),
        nn.Linear(16, 1)
    )

# Універсальна функція для навчання та отримання передбачень
def train_and_eval(model, weight_decay=0.0):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=weight_decay)
    epochs = 50
    
    for epoch in range(epochs):
        model.train()
        outputs = model(X_train_t)
        loss = criterion(outputs, y_train_t)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test_t)
    return test_outputs.numpy()

# 1. Без регуляризації
model_no_reg = create_model()
y_pred_no_reg = train_and_eval(model_no_reg, weight_decay=0.0)
mae_no_reg = mean_absolute_error(y_test, y_pred_no_reg)
r2_no_reg = r2_score(y_test, y_pred_no_reg)

# 2. З регуляризацією L2 (weight_decay=1e-4)
model_reg = create_model()
y_pred_reg = train_and_eval(model_reg, weight_decay=1e-4)
mae_reg = mean_absolute_error(y_test, y_pred_reg)
r2_reg = r2_score(y_test, y_pred_reg)

# Таблиця результатів
results_df = pd.DataFrame({
    'Конфігурація': ['Без регуляризації', 'З регуляризацією (L2)'],
    'MAE': [mae_no_reg, mae_reg],
    'R²': [r2_no_reg, r2_reg]
})
display(results_df)

## Завдання 4
Проаналізуйте результати моделі:
- Побудуйте scatter-графік (реальні проти передбачених) для моделі з регуляризацією.
- Додайте діагональ `y = x`.
- Збережіть зображення під іменем `diabetes_healthrisk_analysis.png`.
- Напишіть короткий аналітичний висновок.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_reg, alpha=0.7, color='blue', label='Predicted vs Real (L2)')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Ідеал: $y = x$')

plt.xlabel('Реальні значення (y_test)')
plt.ylabel('Передбачені значення (y_pred)')
plt.title('Порівняння реальних і передбачених значень (L2-регуляризація)')
plt.legend()
plt.grid(True)
plt.savefig('diabetes_healthrisk_analysis.png')
plt.show()

### Висновок:
1. **Вплив регуляризації:** L2-регуляризація (через `weight_decay`) допомагає обмежити зростання ваг під час навчання (штрафуючи великі значення), що може зробити навчання більш стабільним та покращити здатність до узагальнення (генералізації). Іноді при невеликій кількості епох для простих даних оцінки можуть бути схожими, але регуляризація знижує перенавчання (overfitting).
2. **Вплив ознак:** Найсильніше на показник захворювання зазвичай впливають індекс маси тіла (BMI) та базовий тиск. Залежність на scatter-графіку показує, що модель хоч і не ідеальна (окремі точки відхиляються від діагоналі), але основний тренд прогнозованих значень узгоджується із реальними.